# YOLOv11 Colab 断点续训 Notebook

用途：
- 从 GitHub 拉取项目代码
- 从 Google Drive 解压训练数据
- 生成 Colab 专用数据集配置
- 启动正式训练或从 `last.pt` 断点续训


In [ ]:
REPO_URL = "https://github.com/ailuckly/mengYao.git"
BRANCH = "main"
WORKSPACE = "/content/workspace"
PROJECT_DIR = f"{WORKSPACE}/project"

DRIVE_ROOT = "/content/drive/MyDrive/MengYao_Colab"
DATASET_ZIP = f"{DRIVE_ROOT}/final_dataset_v1_bundle.zip"
TRAIN_RUN_ROOT = f"{DRIVE_ROOT}/train_runs"

RUN_NAME = "colab_pro_formal_v1"
WEIGHTS = "yolo11s.pt"
EPOCHS = 100
BATCH = 32
IMGSZ = 640
DEVICE = "0"

# 首次训练设为 False；中断后改成 True，并把路径改成 Drive 中的 last.pt
RESUME = False
RESUME_PATH = f"{TRAIN_RUN_ROOT}/{RUN_NAME}/weights/last.pt"


In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import shutil
from pathlib import Path

workspace_path = Path(WORKSPACE)
project_path = Path(PROJECT_DIR)
workspace_path.mkdir(parents=True, exist_ok=True)
os.chdir("/content")

if project_path.exists():
    shutil.rmtree(project_path)

!git clone --depth 1 --branch {BRANCH} {REPO_URL} {PROJECT_DIR}
%cd {PROJECT_DIR}

In [ ]:
import zipfile
from pathlib import Path

dataset_zip = Path(DATASET_ZIP)
target_root = Path(PROJECT_DIR)
target_dataset = target_root / "data" / "splits" / "final_v1"

if not dataset_zip.exists():
    raise FileNotFoundError(f"dataset zip not found: {dataset_zip}")

if not target_dataset.exists():
    with zipfile.ZipFile(dataset_zip, "r") as zip_ref:
        zip_ref.extractall(target_root)

print(target_dataset)
!find /content/workspace/project/data/splits/final_v1 -maxdepth 2 -type d | sort

In [ ]:
from pathlib import Path

yaml_path = Path(PROJECT_DIR) / "model" / "configs" / "final_dataset_v1_colab.yaml"
yaml_text = """path: /content/workspace/project/data/splits/final_v1
train: train/images
val: val/images
test: test/images

names:
  0: 银皇后类
  1: 网纹凤梨类
  2: 绿萝类
  3: 心叶蔓绿绒类
  4: Rhaphidophora类
  5: 金钱树类
  6: 叶斑类病害
  7: 枯萎/疫病类病害
  8: 白粉/霉变类病害
  9: 黄化/锈病/虫害类异常
"""
yaml_path.write_text(yaml_text, encoding="utf-8")
print(yaml_path)
!sed -n '1,40p' /content/workspace/project/model/configs/final_dataset_v1_colab.yaml

In [ ]:
%cd /content/workspace/project
!pip install -q -r requirements.txt

In [ ]:
import shlex
import subprocess
from pathlib import Path

Path(TRAIN_RUN_ROOT).mkdir(parents=True, exist_ok=True)

if RESUME:
    cmd = [
        "python", "model/scripts/train.py",
        "--resume",
        "--resume-path", RESUME_PATH,
    ]
else:
    cmd = [
        "python", "model/scripts/train.py",
        "--data", "/content/workspace/project/model/configs/final_dataset_v1_colab.yaml",
        "--weights", WEIGHTS,
        "--device", DEVICE,
        "--epochs", str(EPOCHS),
        "--batch", str(BATCH),
        "--imgsz", str(IMGSZ),
        "--project", TRAIN_RUN_ROOT,
        "--name", RUN_NAME,
        "--optimizer", "auto",
        "--patience", "30",
        "--workers", "4",
        "--save-period", "10",
        "--close-mosaic", "10",
        "--seed", "42",
        "--cos-lr",
    ]

print(" ".join(shlex.quote(part) for part in cmd))
subprocess.run(cmd, check=True)

In [ ]:
!find /content/drive/MyDrive/MengYao_Colab/train_runs -name best.pt
!find /content/drive/MyDrive/MengYao_Colab/train_runs -name last.pt